# Simple Batch Injection Demo (10 x 1000)

This notebook runs batch injection in a simple way:

1. Configure one run
2. Load one Rubin image
3. Define a detection function (MCI example)
4. Run `run_batch(n_iterations=10, n_per_iter=1000)`
5. Analyze combined completeness
6. Save plots from one `make_plots` call

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent
if not (repo_root / 'src').exists():
    repo_root = Path.cwd().resolve()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lsst.daf.butler import Butler
from src.config import InjectionConfig, ClusterConfig
from src.pipeline import InjectionPipeline
from src.detection import run_cluster_detection
from src.plotting import plot_per_iteration_positions, plot_per_iteration_2d
from src.retrieval import ClusterRetrieval

## 1. Configure Batch Run

In [ ]:
BUTLER_REPO = 'dp02'
BUTLER_COLLECTIONS = '2.2i/runs/DP0.2'
DATA_ID = {'tract': 3828, 'patch': 24}

config = InjectionConfig(
    run_name='simple_batch_demo',
    band='i',
    cutout_size=1200,
    n_clusters=1000,
    seed=42,
    edge_buffer=50,
    use_actual_psf=True,
    psf_fwhm_fallback=3.5,
    output_dir=str(repo_root / 'plots' / 'simple_batch_demo'),
    cluster_config=ClusterConfig(
        profile_type='king',
        mag_min=20.0,
        mag_max=26.0,
        r_half_min=2.0,
        r_half_max=10.0,
        concentration_min=5.0,
        concentration_max=30.0,
    ),
)

config

## 2. Load Rubin Image

In [ ]:
butler = Butler(BUTLER_REPO, collections=BUTLER_COLLECTIONS)
pipe = InjectionPipeline(config)
pipe.load_data(butler=butler, data_id=DATA_ID)

bbox_x_min, bbox_y_min = pipe.bboxes[config.band]
psf_obj = pipe.psf_objs[config.band]

print(f'Loaded image shape: {pipe.image.shape}')

## 3. Detection Function (Front-End Example)

Users can replace this function with their own detector.

Required output: `list[dict]` with at least `x` and `y`.

In [ ]:
def mci_detector(image):
    return run_cluster_detection(
        image=image,
        psf_fwhm=config.psf_fwhm_fallback,
        threshold_sigma=3.0,
        mci_max=0.85,
        snr_min=3.0,
        r_half_min=1.0,
        ellipticity_max=0.6,
        box_size=64,
        pixel_scale=config.pixel_scale,
        use_multiscale=True,
        use_mci=True,
        verbose=False,
    )

## 4. Run Batch Injection (10 Iterations x 1000 Clusters)

In [ ]:
iterations = pipe.run_batch(
    n_iterations=10,
    n_per_iter=1000,
    psf_obj=psf_obj,
    bbox_x_min=bbox_x_min,
    bbox_y_min=bbox_y_min,
    psf_fwhm_fallback=config.psf_fwhm_fallback,
    detector_fn=mci_detector,
    store_images=False,
    checkpoint_dir=str(repo_root / 'plots' / 'simple_batch_demo' / 'checkpoints'),
    n_workers=1,
    use_psf_cache=True,
    psf_cache_grid=8,
    psf_cache_size=2000,
    band=config.band,
    verbose=True,
)

print(f'Iterations: {len(iterations)}')
print(f'Total injected: {len(pipe.injection_info)}')
print(f'Total detections: {len(pipe.detection_catalog)}')

## 5. Combined Retrieval And Completeness

In [ ]:
stats = pipe.analyze(match_radius=5.0)
stats

## 6. Save Plots From One API Call

This saves summary/completeness/PSF plots and always saves postage stamps.

In [ ]:
results = pipe.make_plots(
    plots=['before_after', 'completeness_1d', 'completeness_2d', 'psf_fwhm_hist'],
    show=True,
    save=True,
    poster_style=False,
    n_stamps=6,
)

print('Saved files:')
for k, v in results['saved'].items():
    print(f' - {k}: {v}')

## 7. Optional Per-Iteration Views

These help inspect iteration-to-iteration behavior in the batch run.

In [ ]:
_ = plot_per_iteration_positions(
    iterations=iterations,
    base_image=pipe.image,
    ClusterRetrieval=ClusterRetrieval,
    n_show=4,
    match_radius=5.0,
)

_ = plot_per_iteration_2d(
    iterations=iterations,
    config=config,
    ClusterRetrieval=ClusterRetrieval,
    n_show=4,
    match_radius=5.0,
)